In [0]:
#CONNECTION SETUP
server   = "tijartik-sql-server.database.windows.net"
database = "tijartek"
username = "Yasmeen"
password = "Y_dataengineer2026TIJarTIK"

jdbc_url = f"jdbc:sqlserver://{server}:1433;database={database};user={username};password={password};encrypt=true;trustServerCertificate=false;"
print(" Connection configured")

 Connection configured


In [0]:
#CREATE BRONZE DATABASE
spark.sql("CREATE DATABASE IF NOT EXISTS tijartek_bronze")
print(" tijartek_bronze database ready")

 tijartek_bronze database ready


In [0]:
# INGEST ALL 15 TABLES INTO BRONZE
 
from datetime import datetime
from pyspark.sql.functions import lit
 
# All source tables
tables = [
    "Locations",
    "Categories",
    "Sellers",
    "Customers",
    "Products",
    "Promotions",
    "Sessions",
    "User_Events",
    "[Orders]",
    "Order_Items",
    "Payments",
    "Shipments",
    "[Returns]",
    "Reviews",
    "Inventory_Logs"
]
 
ingestion_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
 
for table in tables:
    clean_name = table.replace("[", "").replace("]", "").lower()
    
    print(f" Loading {clean_name}...")
    
    # Read from Azure SQL
    df = spark.read \
        .format("jdbc") \
        .option("url", jdbc_url) \
        .option("dbtable", f"dbo.{table}") \
        .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver") \
        .load()
    
    # Add bronze metadata columns
    df = df \
        .withColumn("_source", lit("azure_sql")) \
        .withColumn("_ingested_at", lit(ingestion_time)) \
        .withColumn("_source_table", lit(clean_name))
    
    # Write to Delta table in tijartek_bronze
    df.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(f"tijartek_bronze.{clean_name}")
    
    count = spark.table(f"tijartek_bronze.{clean_name}").count()
    print(f" {clean_name}: {count:,} rows loaded")
 
print("\n BRONZE LAYER COMPLETE!")

 Loading locations...
 locations: 50 rows loaded
 Loading categories...
 categories: 25 rows loaded
 Loading sellers...
 sellers: 1,000 rows loaded
 Loading customers...
 customers: 80,000 rows loaded
 Loading products...
 products: 50,000 rows loaded
 Loading promotions...
 promotions: 500 rows loaded
 Loading sessions...
 sessions: 120,000 rows loaded
 Loading user_events...
 user_events: 500,000 rows loaded
 Loading orders...
 orders: 150,000 rows loaded
 Loading order_items...
 order_items: 338,051 rows loaded
 Loading payments...
 payments: 150,000 rows loaded
 Loading shipments...
 shipments: 131,981 rows loaded
 Loading returns...
 returns: 26,726 rows loaded
 Loading reviews...
 reviews: 60,000 rows loaded
 Loading inventory_logs...
 inventory_logs: 100,000 rows loaded

 BRONZE LAYER COMPLETE!


In [0]:
# VERIFY BRONZE LAYER
 
print("=" * 50)
print("BRONZE LAYER - ROW COUNTS")
print("=" * 50)
 
bronze_tables = [
    "locations", "categories", "sellers", "customers", "products",
    "promotions", "sessions", "user_events", "orders", "order_items",
    "payments", "shipments", "returns", "reviews", "inventory_logs"
]
 
total_rows = 0
for table in bronze_tables:
    count = spark.table(f"tijartek_bronze.{table}").count()
    total_rows += count
    print(f"{table:20} {count:>10,} rows")
 
print("-" * 40)
print(f"{'TOTAL':20} {total_rows:>10,} rows")
print("=" * 50)

BRONZE LAYER - ROW COUNTS
locations                    50 rows
categories                   25 rows
sellers                   1,000 rows
customers                80,000 rows
products                 50,000 rows
promotions                  500 rows
sessions                120,000 rows
user_events             500,000 rows
orders                  150,000 rows
order_items             338,051 rows
payments                150,000 rows
shipments               131,981 rows
returns                  26,726 rows
reviews                  60,000 rows
inventory_logs          100,000 rows
----------------------------------------
TOTAL                 1,708,333 rows


In [0]:
# QUICK DATA PREVIEW

 
# Preview each table - first 3 rows
for table in bronze_tables:
    print(f"\n {table.upper()}")
    spark.table(f"tijartek_bronze.{table}").show(3, truncate=True)


 LOCATIONS
+-----------+----------+----------+---------+-------------------+-------------+
|location_id|      city|    region|  _source|       _ingested_at|_source_table|
+-----------+----------+----------+---------+-------------------+-------------+
|          1|     Cairo|     Cairo|azure_sql|2026-05-21 23:45:35|    locations|
|          2|Alexandria|Alexandria|azure_sql|2026-05-21 23:45:35|    locations|
|          3|      Giza|      Giza|azure_sql|2026-05-21 23:45:35|    locations|
+-----------+----------+----------+---------+-------------------+-------------+
only showing top 3 rows

 CATEGORIES
+-----------+-------------------+---------------+---------+-------------------+-------------+
|category_id|               name|commission_rate|  _source|       _ingested_at|_source_table|
+-----------+-------------------+---------------+---------+-------------------+-------------+
|          1|        Electronics|   8.5000000000|azure_sql|2026-05-21 23:45:35|   categories|
|          2|  